# 02 — DDPM: Denoising Diffusion Probabilistic Models

DDPM defines a fixed forward noising process and learns to reverse it.

## Forward and Reverse Process

**Forward process**: gradually add Gaussian noise over *T* steps:
$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} x_{t-1}, \beta_t I)$$
Using the reparameterisation trick, we can sample *any* noise level directly:
$$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$
where $\bar{\alpha}_t = \prod_{s=1}^t (1-\beta_s)$.

**Noise schedule**: a linear schedule $\beta_t$ from $\beta_1=10^{-4}$ to $\beta_T=0.02$ over $T=1000$ steps works well for images.

**Reverse process**: a neural network (UNet) predicts the noise $\epsilon$ added at step *t*:
$$p_\theta(x_{t-1}|x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \sigma_t^2 I)$$

**Training objective**: simple noise prediction loss:
$$\mathcal{L} = \mathbb{E}_{t, x_0, \epsilon}[||\epsilon - \epsilon_\theta(x_t, t)||^2]$$

**Sampling**: start from $x_T \sim \mathcal{N}(0, I)$, iteratively denoise from $t=T$ to $t=0$.

In [ ]:
# DDPM from scratch on 1D data
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Noise schedule
T = 200
betas = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def q_sample(x0, t, noise=None):
    """Sample from forward process at step t."""
    if noise is None:
        noise = torch.randn_like(x0)
    sqrt_alpha_bar = alpha_bars[t].sqrt()
    sqrt_one_minus_alpha_bar = (1 - alpha_bars[t]).sqrt()
    return sqrt_alpha_bar * x0 + sqrt_one_minus_alpha_bar * noise, noise

# 1D dataset: bimodal
n = 2000
x_data = torch.cat([torch.randn(n//2) * 0.3 + 2, torch.randn(n//2) * 0.3 - 2]).unsqueeze(1)

# Noise schedule visualisation
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for k, t_val in enumerate([0, 50, 100, 199]):
    x_noisy, _ = q_sample(x_data[:200], t_val)
    axes[k].hist(x_noisy.squeeze().numpy(), bins=30, density=True, alpha=0.7)
    axes[k].set_title(f't={t_val}')
    axes[k].set_xlim(-5, 5)
plt.suptitle('Forward Noising Process')
plt.tight_layout()
plt.savefig('/tmp/ddpm_forward.png', dpi=100, bbox_inches='tight')
plt.show()
print('Forward process visualisation saved')

In [ ]:
# Simple noise prediction network
class NoiseNet(nn.Module):
    """1D noise predictor with time embedding."""
    def __init__(self, data_dim=1, hidden=64, T=200):
        super().__init__()
        self.time_embed = nn.Embedding(T, hidden)
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, data_dim),
        )

    def forward(self, x, t):
        t_emb = self.time_embed(t)  # (batch, hidden)
        h = torch.cat([x, t_emb], dim=-1)
        return self.net(h)

noise_net = NoiseNet(data_dim=1, T=T)
opt = torch.optim.Adam(noise_net.parameters(), lr=3e-4)

# Training loop
for epoch in range(2000):
    idx = torch.randint(0, len(x_data), (256,))
    x0 = x_data[idx]
    t = torch.randint(0, T, (256,))
    x_t, eps = q_sample(x0, t)
    eps_pred = noise_net(x_t, t)
    loss = ((eps_pred - eps)**2).mean()
    opt.zero_grad(); loss.backward(); opt.step()

    if epoch % 500 == 0:
        print(f'Epoch {epoch}: loss={loss.item():.4f}')

print('Training complete')

In [ ]:
# DDPM sampling (reverse process)
@torch.no_grad()
def ddpm_sample(model, n_samples, T, betas, alphas, alpha_bars):
    x = torch.randn(n_samples, 1)
    for t in range(T - 1, -1, -1):
        t_batch = torch.full((n_samples,), t, dtype=torch.long)
        eps_pred = model(x, t_batch)

        # Reverse step
        alpha_t = alphas[t]
        alpha_bar_t = alpha_bars[t]
        beta_t = betas[t]

        coef1 = 1 / alpha_t.sqrt()
        coef2 = beta_t / (1 - alpha_bar_t).sqrt()
        mean = coef1 * (x - coef2 * eps_pred)

        if t > 0:
            noise = torch.randn_like(x)
            x = mean + beta_t.sqrt() * noise
        else:
            x = mean
    return x

samples = ddpm_sample(noise_net, 500, T, betas, alphas, alpha_bars)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(x_data.squeeze().numpy(), bins=40, density=True, alpha=0.7, label='Real')
ax1.set_title('Real data')
ax2.hist(samples.squeeze().numpy(), bins=40, density=True, alpha=0.7, color='orange', label='DDPM')
ax2.set_title('DDPM samples')
for ax in [ax1, ax2]: ax.set_xlim(-5, 5)
plt.suptitle('DDPM: Real vs Generated')
plt.tight_layout()
plt.savefig('/tmp/ddpm_samples.png', dpi=100, bbox_inches='tight')
plt.show()
print('DDPM sampling complete')